In [ ]:


import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np


# Model

def classification_model():

    model = models.Sequential([

        layers.Conv2D(
            32,
            (3,3),
            activation='relu',
            input_shape=(128,128,3)
        ),

        layers.MaxPooling2D((2,2)),

        layers.Conv2D(
            64,
            (3,3),
            activation='relu'
        ),

        layers.MaxPooling2D((2,2)),

        layers.Conv2D(
            128,
            (3,3),
            activation='relu'
        ),

        layers.GlobalAveragePooling2D(),

        layers.Dense(
            128,
            activation='relu'
        ),

        layers.Dense(
            4,
            activation='softmax'
        )

    ])

    return model


# Build

model = classification_model()


# Optimizer

optimizer = tf.keras.optimizers.Adam(1e-4)


# Data

x_train = np.random.rand(
    500,
    128,
    128,
    3
)

y_train = tf.keras.utils.to_categorical(
    np.random.randint(0,4,500),
    num_classes=4
)


# Compile

model.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


# Train

model.fit(
    x_train,
    y_train,
    epochs=10,
    batch_size=16
)


# Teacher

teacher_model = tf.keras.models.clone_model(model)

teacher_model.set_weights(
    model.get_weights()
)


# Save

model.save("phase1_classification_lwf.h5")


In [ ]:


import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np


# Model

def segmentation_model():

    inputs = layers.Input(
        shape=(128,128,3)
    )

    c1 = layers.Conv2D(
        32,
        3,
        activation='relu',
        padding='same'
    )(inputs)

    p1 = layers.MaxPooling2D()(c1)

    c2 = layers.Conv2D(
        64,
        3,
        activation='relu',
        padding='same'
    )(p1)

    p2 = layers.MaxPooling2D()(c2)

    b1 = layers.Conv2D(
        128,
        3,
        activation='relu',
        padding='same'
    )(p2)

    u1 = layers.Conv2DTranspose(
        64,
        2,
        strides=2,
        padding='same'
    )(b1)

    u1 = layers.concatenate([u1,c2])

    c3 = layers.Conv2D(
        64,
        3,
        activation='relu',
        padding='same'
    )(u1)

    u2 = layers.Conv2DTranspose(
        32,
        2,
        strides=2,
        padding='same'
    )(c3)

    u2 = layers.concatenate([u2,c1])

    c4 = layers.Conv2D(
        32,
        3,
        activation='relu',
        padding='same'
    )(u2)

    outputs = layers.Conv2D(
        1,
        1,
        activation='sigmoid'
    )(c4)

    model = models.Model(
        inputs,
        outputs
    )

    return model


# Build

model = segmentation_model()


# Optimizer

optimizer = tf.keras.optimizers.Adam(1e-4)


# Alpha

ALPHA = 0.5


# Loss

def lwf_loss(y_true, y_pred, old_pred):

    seg_loss = tf.reduce_mean(
        tf.keras.losses.binary_crossentropy(
            y_true,
            y_pred
        )
    )

    distill_loss = tf.reduce_mean(
        tf.square(old_pred - y_pred)
    )

    total_loss = (
        ALPHA * seg_loss
    ) + (
        (1 - ALPHA) * distill_loss
    )

    return total_loss


# Data

x_seg = np.random.rand(
    300,
    128,
    128,
    3
)

y_seg = np.random.randint(
    0,
    2,
    (300,128,128,1)
)


# Teacher output

old_predictions = teacher_model.predict(
    x_seg,
    batch_size=8
)


# Train

EPOCHS = 10
BATCH_SIZE = 8

for epoch in range(EPOCHS):

    for i in range(
        0,
        len(x_seg),
        BATCH_SIZE
    ):

        x_batch = x_seg[i:i+BATCH_SIZE]

        y_batch = y_seg[i:i+BATCH_SIZE]

        old_batch = old_predictions[
            i:i+BATCH_SIZE
        ]

        with tf.GradientTape() as tape:

            preds = model(x_batch)

            loss = lwf_loss(
                y_batch,
                preds,
                old_batch
            )

        grads = tape.gradient(
            loss,
            model.trainable_variables
        )

        optimizer.apply_gradients(
            zip(
                grads,
                model.trainable_variables
            )
        )

    print(
        "Epoch:",
        epoch + 1,
        "Loss:",
        loss.numpy()
    )


# Teacher

teacher_model = tf.keras.models.clone_model(model)

teacher_model.set_weights(
    model.get_weights()
)


# Save

model.save("phase2_segmentation_lwf.h5")


In [ ]:


import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np


# Model

def prediction_model():

    model = models.Sequential([

        layers.Conv2D(
            32,
            (3,3),
            activation='relu',
            input_shape=(128,128,3)
        ),

        layers.MaxPooling2D((2,2)),

        layers.Conv2D(
            64,
            (3,3),
            activation='relu'
        ),

        layers.MaxPooling2D((2,2)),

        layers.Conv2D(
            128,
            (3,3),
            activation='relu'
        ),

        layers.GlobalAveragePooling2D(),

        layers.Dense(
            128,
            activation='relu'
        ),

        layers.Dense(
            1,
            activation='linear'
        )

    ])

    return model


# Build

model = prediction_model()


# Optimizer

optimizer = tf.keras.optimizers.Adam(1e-4)


# Alpha

ALPHA = 0.5


# Loss

def lwf_loss(y_true, y_pred, old_pred):

    pred_loss = tf.reduce_mean(
        tf.keras.losses.mse(
            y_true,
            y_pred
        )
    )

    distill_loss = tf.reduce_mean(
        tf.square(old_pred - y_pred)
    )

    total_loss = (
        ALPHA * pred_loss
    ) + (
        (1 - ALPHA) * distill_loss
    )

    return total_loss


# Data

x_pred = np.random.rand(
    300,
    128,
    128,
    3
)

y_pred = np.random.rand(
    300,
    1
)


# Teacher output

old_predictions = teacher_model.predict(
    x_pred,
    batch_size=16
)


# Train

EPOCHS = 10
BATCH_SIZE = 16

for epoch in range(EPOCHS):

    for i in range(
        0,
        len(x_pred),
        BATCH_SIZE
    ):

        x_batch = x_pred[i:i+BATCH_SIZE]

        y_batch = y_pred[i:i+BATCH_SIZE]

        old_batch = old_predictions[
            i:i+BATCH_SIZE
        ]

        with tf.GradientTape() as tape:

            preds = model(x_batch)

            loss = lwf_loss(
                y_batch,
                preds,
                old_batch
            )

        grads = tape.gradient(
            loss,
            model.trainable_variables
        )

        optimizer.apply_gradients(
            zip(
                grads,
                model.trainable_variables
            )
        )

    print(
        "Epoch:",
        epoch + 1,
        "Loss:",
        loss.numpy()
    )


# Save

model.save("phase3_prediction_lwf.h5")